In [ ]:
!nvidia-smi

Tue Feb 24 20:08:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   42C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
REPO_URL = "https://github.com/ceus90/CS224N-TheEfficiencyThreshold.git"
REPO_DIR = "CS224N-TheEfficiencyThreshold"

!git clone {REPO_URL}
%cd {REPO_DIR}

Cloning into 'CS224N-TheEfficiencyThreshold'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 208 (delta 91), reused 171 (delta 73), pack-reused 0 (from 0)
Receiving objects: 100% (208/208), 1.84 MiB | 26.58 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/CS224N-TheEfficiencyThreshold


In [ ]:
# switch to abi_splits branch
!git checkout abi_splits
!git pull

Branch 'abi_splits' set up to track remote branch 'abi_splits' from 'origin'.
Switched to a new branch 'abi_splits'
Already up to date.


In [ ]:
!ls data/splits_out/gsm8k

manifest.json	  N16_seed0.jsonl   N256_seed1.jsonl  N32_seed2.jsonl
N128_seed0.jsonl  N16_seed1.jsonl   N256_seed2.jsonl  N64_seed0.jsonl
N128_seed1.jsonl  N16_seed2.jsonl   N32_seed0.jsonl   N64_seed1.jsonl
N128_seed2.jsonl  N256_seed0.jsonl  N32_seed1.jsonl   N64_seed2.jsonl


In [ ]:
# Install libraries
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets accelerate peft evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

SPLITS_DIR = Path("data/splits_out/gsm8k")

assert SPLITS_DIR.exists(), f"Missing splits dir: {SPLITS_DIR}"
print("Found split files:", sorted([p.name for p in SPLITS_DIR.glob("*.jsonl")])[:10])

Found split files: ['N128_seed0.jsonl', 'N128_seed1.jsonl', 'N128_seed2.jsonl', 'N16_seed0.jsonl', 'N16_seed1.jsonl', 'N16_seed2.jsonl', 'N256_seed0.jsonl', 'N256_seed1.jsonl', 'N256_seed2.jsonl', 'N32_seed0.jsonl']


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import time
import re
import torch
from datasets import load_dataset
from tqdm import tqdm
import numpy as np

# 1. Load GSM8K Test Set
print("Loading GSM8K test set...")
ds = load_dataset("gsm8k", "main", split="test")

# 2. Define Helper Functions
def extract_answer_number(text):
    """
    Extracts the last number from the text.
    GSM8K usually puts the answer after '#### '.
    """
    # Look for the specific GSM8K marker first
    if "####" in text:
        text = text.split("####")[-1]

    # Find all numbers (integers or floats)
    # This regex handles commas like 1,000 and decimals like 1.5
    text = text.replace(',', '')
    matches = re.findall(r'-?\d+\.?\d*', text)

    if matches:
        return matches[-1]
    return None

def get_peak_vram_mb():
    return torch.cuda.max_memory_allocated() / (1024 ** 2)

# 3. Evaluation Config
# Limit eval_size if you want a quick test, else set to len(ds)
EVAL_SIZE = len(ds)
# EVAL_SIZE = 50 # Uncomment for a quick smoke test

print(f"Starting evaluation on {EVAL_SIZE} examples...")

Loading GSM8K test set...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Starting evaluation on 1319 examples...


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import re
import time

# --- Config ---
MODEL_NAME = "Qwen/Qwen3-8B"
BATCH_SIZE = 4          # Increased since 4-bit uses less VRAM
MAX_NEW_TOKENS = 512    # Kept at 512 for safety on harder reasoning, but can lower to 256
EVAL_SIZE = len(ds)     # Run full test set

In [ ]:
# --- Load Model with 4-bit Quantization & SDPA ---
print("Loading model with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa", # Switched to 'sdpa' (native torch fast attention) to avoid install issues
    device_map="auto",
)
model.eval()

print("Model loaded successfully!")

Loading model with 4-bit quantization...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder for our results if it doesn't exist
DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/qwen_8b_GSM8K_Eval"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print(f"Saving checkpoints to: {DRIVE_SAVE_DIR}")

Mounted at /content/drive
Saving checkpoints to: /content/drive/MyDrive/Colab_Data/qwen_8b_GSM8K_Eval


In [ ]:
import os
import json

# --- Checkpoint Setup ---
# UPDATE: Point to Google Drive path
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "qwen_8b_gsm8k_eval_checkpoint.jsonl")
results = []

# Resume if checkpoint exists
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint file: {CHECKPOINT_FILE}. Loading previous results...")
    with open(CHECKPOINT_FILE, "r") as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Resuming from example {len(results)}.")
else:
    print(f"No checkpoint found at {CHECKPOINT_FILE}. Starting from scratch.")

# Helper for answer extraction
def extract_answer_number(text):
    if "####" in text:
        text = text.split("####")[-1]
    text = text.replace(',', '')
    matches = re.findall(r'-?\d+\.?\d*', text)
    return matches[-1] if matches else None

# Prepare dataset
subset = ds.select(range(EVAL_SIZE))
questions = subset["question"]
ground_truths = [extract_answer_number(a) for a in subset["answer"]]

start_idx = len(results)

print(f"Starting batched evaluation (Batch Size: {BATCH_SIZE})...")

# --- Batched Evaluation Loop ---
for i in tqdm(range(start_idx, EVAL_SIZE, BATCH_SIZE)):
    batch_questions = questions[i : i + BATCH_SIZE]
    batch_ground_truths = ground_truths[i : i + BATCH_SIZE]

    # Prepare prompts
    batch_texts = [
        tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
        for q in batch_questions
    ]

    inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    # Generate
    batch_start = time.time()
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    batch_time = time.time() - batch_start

    # Decode
    new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
    decoded_outputs = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    # Process Batch Results
    batch_results = []
    for j, response in enumerate(decoded_outputs):
        pred = extract_answer_number(response)
        gt = batch_ground_truths[j]

        is_correct = False
        if pred is not None and gt is not None:
            if float(pred) == float(gt):
                is_correct = True

        # Store detailed result
        res_item = {
            "question_idx": i + j,
            "prediction": pred,
            "ground_truth": gt,
            "is_correct": is_correct,
            "is_unknown": (pred is None),
            "new_tokens": int(new_tokens[j].numel()),
            "batch_time_share": batch_time / len(batch_questions) # Approx time per sample
        }
        batch_results.append(res_item)
        results.append(res_item)

    # Save to file immediately (Checkpointing)
    with open(CHECKPOINT_FILE, "a") as f:
        for item in batch_results:
            f.write(json.dumps(item) + "\n")

# --- Final Metrics Calculation ---
# We recalculate everything from the full 'results' list
correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
total_new_tokens = sum(r["new_tokens"] for r in results)
total_gen_time_sum = sum(r["batch_time_share"] for r in results)

final_metrics = {
    "method": "icl_0_shot",
    "model": MODEL_NAME,
    "eval_size": len(results),
    "accuracy": correct_count / len(results) if results else 0,
    "unknown_frac": unknown_count / len(results) if results else 0,
    "gen_tokens_per_sec": total_new_tokens / total_gen_time_sum if total_gen_time_sum > 0 else 0,
    "total_gen_time_min": total_gen_time_sum / 60
}

print("\nFinal Metrics:")
for k, v in final_metrics.items():
    print(f"{k}: {v}")

Found checkpoint file: /content/drive/MyDrive/Colab_Data/qwen_8b_GSM8K_Eval/qwen_8b_gsm8k_eval_checkpoint.jsonl. Loading previous results...
Resuming from example 208.
Starting batched evaluation (Batch Size: 4)...



  0%|          | 0/278 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

100%|██████████| 278/278 [5:32:46<00:00, 71.82s/it]


Final Metrics:
method: icl_0_shot
model: Qwen/Qwen3-8B
eval_size: 1319
accuracy: 0.24260803639120546
unknown_frac: 0.000758150113722517
gen_tokens_per_sec: 28.470345421136223
total_gen_time_min: 395.340010813872


In [ ]:
# --- Debugging: Inspect First Example ---
example = ds[0]
question = example["question"]
ground_truth = example["answer"]

print("Question:\n", question)
print("\nGround Truth:\n", ground_truth)

# Prepare input
messages = [{"role": "user", "content": question}]
text_input = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\nFormatted Prompt:\n", text_input)

model_inputs = tokenizer([text_input], return_tensors="pt").to(model.device)

# Generate
with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id
    )

# Decode
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\nModel Response:\n", response)

# Check Extraction
pred_num = extract_answer_number(response)
gt_num = extract_answer_number(ground_truth)

print(f"\nExtracted Prediction: {pred_num}")
print(f"Extracted Ground Truth: {gt_num}")
print(f"Match: {float(pred_num) == float(gt_num) if pred_num and gt_num else 'False'}")

Question:
 Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Ground Truth:
 Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

Formatted Prompt:
 <|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|im_end|>
<|im_start|>assistant


Model Response:
 <think>
Okay, let me try to figure out how much Janet makes at the farmers' market every day. So, first, let me break down the problem step by step. 

Janet's ducks lay 16 eggs per day. That's 